# TrainJudge on a free GPU

Should you fine-tune at all, and did it actually work? This notebook runs TrainJudge end to end
on Colab's free NVIDIA GPU, with the PyTorch backend:

1. **Diagnose** two healthcare goals: one that shouldn't be fine-tuned, one that should
2. **Mask** the patient identifiers the audit finds, and **train** a LoRA adapter
3. **Verify** it on held-out data and a general-capability check, and read the report

**Before you start:** Runtime → Change runtime type → **T4 GPU**. The whole notebook takes about
15–20 minutes. All data in it is fictional.

In [ ]:
import platform
import shutil
import subprocess

if platform.system() == "Darwin":
    print("This notebook is running on your Mac, not on Colab.")
    print("Select Kernel -> Colab -> new server -> GPU (T4), then run this cell again.")
elif shutil.which("nvidia-smi"):
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    ).stdout.strip()
    print(f"GPU ready: {gpu}")
else:
    print("Connected to a Colab server without a GPU (CPU or TPU).")
    print("TrainJudge needs an NVIDIA GPU here: remove this server (Colab: Remove Server),")
    print("then connect a new one with GPU -> T4, and run this cell again.")

## 1. Install

Clones the repo (for the demo datasets) and installs TrainJudge with the PyTorch backend. Colab
already has PyTorch with CUDA.

*While the repository is private,* set `GITHUB_TOKEN` to a token with read access when prompted,
or leave it empty once the repo is public.

In [ ]:
import getpass
import os
import subprocess

token = getpass.getpass("GitHub token (leave empty if the repo is public): ").strip()
url = (
    f"https://{token}@github.com/Himanshukurrey/trainjudge"
    if token
    else "https://github.com/Himanshukurrey/trainjudge"
)
subprocess.run(["git", "clone", "--quiet", "--depth", "1", url, "trainjudge"], check=True)
os.chdir("trainjudge")
!pip install -q -e ".[cuda]"
!trainjudge --version

## 2. Diagnose before training

**Case A: answer staff questions from the hospital formulary.** The answers are dosing facts that
change whenever the formulary does, so TrainJudge should recommend retrieval, not fine-tuning.

In [ ]:
!trainjudge diagnose --dataset demo/domains/healthcare/formulary_faq/data.jsonl --model Qwen3-0.6B \
  --goal "answer staff questions about medication dosing from our hospital formulary"

**Case B: extract diagnoses and ICD-10 codes from clinical notes.** A structured-output task,
which fine-tuning suits, but the notes contain patient identifiers.

In [ ]:
!trainjudge diagnose --dataset demo/domains/healthcare/clinical_coding/data.jsonl --model Qwen3-0.6B \
  --goal "extract diagnoses and ICD-10 codes from clinical notes into our JSON format"

## 3. Mask the identifiers and train

`train` refuses data with sensitive identifiers. `--mask-sensitive` replaces them with placeholders
like `[MRN]` and `[DOB]` first. `--replay` mixes the base model's own answers to general prompts
into training, so it keeps its general skills while it learns the task.

In [ ]:
!trainjudge train --dataset demo/domains/healthcare/clinical_coding/data.jsonl --model Qwen3-0.6B \
  --backend torch --mask-sensitive --replay 208 \
  --iters 150 --learning-rate 2e-5 --rank 8 --num-layers 8 \
  --goal "extract diagnoses and ICD-10 codes from clinical notes into our JSON format"

## 4. Verify: did it actually work?

Scores the base model and the fine-tuned adapter on the held-out test split (JSON exact match,
field by field) and on a 60-prompt general-capability suite, then issues **IMPROVED**, **REGRESSED**
or **REJECTED**.

In [ ]:
import glob

run = sorted(glob.glob("trainjudge-runs/*clinical_coding*"))[-1]
print(run)
!trainjudge verify {run}

In [ ]:
from IPython.display import Markdown

Markdown(open(f"{run}/EXPERIMENT_REPORT.md", encoding="utf-8").read())

## Next

- Try other domains in `demo/domains/` (legal, e-commerce, customer support, HR, BFSI, education).
- Train on your own JSONL: `trainjudge diagnose` first, then `train` and `verify`.
- `trainjudge status --all` lists every run in this session.